In [0]:
# Bronze Layer: Raw Customers Ingestion
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col

spark = SparkSession.builder.getOrCreate()

LANDING_PATH  = "s3://ecom-landing-zone-2026/landing/customers/"
CHECKPOINT    = "s3://ecom-landing-zone-2026/checkpoints/customers_checkpoint/"
BRONZE_TABLE  = "ecomstore.ecomlake.bronze_customers"
SOURCE_SYSTEM = "customer_management_system"

# Pre-create table with Auto-Compaction
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        customer_id STRING, first_name STRING, last_name STRING, email STRING, 
        phone STRING, gender STRING, date_of_birth STRING, city STRING, 
        state STRING, pincode STRING, registration_date STRING, customer_tier STRING, 
        is_active STRING, created_at STRING, updated_at STRING, 
        _source_system STRING, _batch_id STRING, _rescued_data STRING,
        _ingestion_timestamp TIMESTAMP, _source_file_name STRING, 
        _pipeline_layer STRING, _source_system_override STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")

# schema hint for schema evolution - all columns will be strings 
# (we'll convert to proper types in downstream) 
customers_schema = """
    customer_id       STRING,
    first_name        STRING,
    last_name         STRING,
    email             STRING,
    phone             STRING,
    gender            STRING,
    date_of_birth     STRING,
    city              STRING,
    state             STRING,
    pincode           STRING,
    registration_date STRING,
    customer_tier     STRING,
    is_active         STRING,
    created_at        STRING,
    updated_at        STRING,
    _source_system    STRING,
    _batch_id         STRING
"""

# 2. Auto Loader Read with Rescued Data Catch
raw_customers_df = (
    spark.readStream
    .format("cloudFiles")                                           # Auto Loader format
    .option("cloudFiles.format", "csv")                             # source file format
    .option("cloudFiles.schemaLocation", CHECKPOINT + "schema/")
    .option("cloudFiles.inferColumnTypes", "false")                 # keep everything as string
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")      # handle schema drift
    .option("header", "true")
    .option("mode", "PERMISSIVE")                                   # allow malformed CSV rows
    .option("cloudFiles.schemaHints", customers_schema)             # convert schema to string for ingestion 
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")        # 🛡️ Traps corrupted CSV rows
    .load(LANDING_PATH)
    # Add ingestion metadata columns
    .withColumn("_ingestion_timestamp", current_timestamp())        # when did we ingest this?
    .withColumn("_source_file_name", col("_metadata.file_path"))    # which file did it come from?
    .withColumn("_pipeline_layer", lit("bronze"))
    .withColumn("_source_system_override", lit(SOURCE_SYSTEM))
)

# Write to Bronze Delta Table (Append Only)
(
    raw_customers_df.writeStream
    .format("delta")
    .outputMode("append")                       # bronze is ALWAYS append-only
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")              # allow schema evolution
    .trigger(availableNow=True)                 # process all available files, then stop
    .toTable(BRONZE_TABLE)                      # writes to managed Delta table
)
print(f"✅ Bronze ingestion complete for: {BRONZE_TABLE}")